In [6]:
import os
import time
import folium
import tomllib
import shapely
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from typing import Union
from pathlib import Path
from dotenv import dotenv_values
from shapely.geometry import Polygon
from shapely import points, contains, prepare

# Setting Config File

As we have seen in previous notebooks, it is possible to make all the checks for the seismic data using similar vectorization techniques (such as polygon, numeric, etc). However, define and work around each one of the possible checks can be annoying and time-consuming. For this reason, we will define a configuration file where we can set all the checks that we want to perform on the seismic data. This way, we can easily modify the checks without having to change the code. We will use a TOML file for this purpose, as it is a simple and human-readable format.

The main idea here is to design a new scheme using a config file (in format TOML) that allows to define the checks that we want to perform on the seismic data. To achieve this, we will define a TOML file and a wrapper function that reads the config file and executes the checks accordingly. This way, we can easily modify the checks without having to change the code.

## Configuration for query data from seiscomp

The first step is to define the configuration for querying data from Seiscomp. This configuration will include the connection parameters for the database, as well as the query that we want to execute to retrieve the seismic data. We will define this configuration in a TOML file, which will be read by our code to establish the connection and execute the query (See TOML schema notebook for more details). The configuration will require the following parameters:

- host: The hostname of the database server.
- port: The port number of the database server.
- user: The username to connect to the database.
- password: The password to connect to the database.
- database: The name of the database to connect to.
- query: The SQL query to execute to retrieve the seismic data.

The recommended way to store the database credentials is to use a .env file, which will be read by our code to set the environment variables. This way, we can keep the credentials secure and avoid hardcoding them in the code or the config file. Moreover, the SQL query can be defined in a separate .sql file, which will be read by our code to execute the query. This way, we can easily modify the query without having to change the code or the config file. The TOML then will reference the .sql file to read the query. This approach allows for a clean separation of concerns and makes it easier to manage and update the configuration for querying data from Seiscomp.

The structure of the TOML file for the Seiscomp configuration will look like this:
```toml
[database]
env_file = ".env"
# Not required if env_file is provided and contains the host variable
host = "localhost"
port = 3306
user = "username"
password = "password"
database = "seiscomp"

[query]
sql_file = "query.sql"
```

If .env file is provided, it must have the following variables:
```
# Credentials for the database connection
DB_HOST=localhost
DB_PORT=3306
DB_USER=username
DB_PASSWORD=password
DB_NAME=seiscomp

# Credentials for any other database connection parameters that might be needed too
DB_HOST2=localhost
DB_PORT2=3306
DB_USER2=username
DB_PASSWORD2=password
DB_NAME2=seiscomp
```

The advantage of using this approach is that we can easily modify the database connection parameters and the SQL query without having to change the code. We can simply update the .sql file or the .env file (adding more environment variables if needed) and the code will read the new configuration and execute the checks accordingly. This makes it easier to manage and maintain the code, as well as to adapt it to different environments or requirements.

## Load config file

First of all, we need to define a function that reads the TOML configuration file and returns a dictionary with the connection parameters and the SQL query. This function will also handle the loading of the .env file and the .sql file, as well as the validation of the configuration. The function will be called `load_config` and it will take the path to the TOML file as an argument. It will return a dictionary with the following structure:

In [7]:
# ──────────────────────────────────────────────────────────────────
# HELPER 1 — Credentials
# ──────────────────────────────────────────────────────────────────

def _load_credentials(
    db_cfg:           dict,
    base_dir:         Path,
    config_name:      str,
    credentials_keys: tuple[str, str, str, str, str],
) -> dict:
    """
    Resolve database credentials from (in priority order):
        1. Live os.environ variables
        2. The declared .env file
        3. Inline values in the TOML [database] section

    Parameters
    ----------
    db_cfg : dict
        Parsed contents of the TOML [database] section.
    base_dir : Path
        Directory of the TOML file, used to resolve relative env_file paths.
    config_name : str
        TOML filename, used only for error messages.
    credentials_keys : tuple[str, str, str, str, str]
        Ordered key names for (host, user, password, database, port).

    Returns
    -------
    dict with keys: host, port, user, password, database.

    Raises
    ------
    FileNotFoundError  – env_file declared but not found on disk.
    KeyError           – required credential absent from all sources.
    TypeError          – a credentials_key is not a string, or port is
                         not coercible to int.
    ValueError         – port is outside the valid 1–65535 range.
    """

    # ── Validate credentials_keys types upfront ────────────────────
    key_labels = ("HOST", "USER", "PWD", "DB", "PORT")
    for i, key in enumerate(credentials_keys):
        if not isinstance(key, str):
            raise TypeError(
                f"{key_labels[i]} key must be a string, "
                f"got {key!r} (type: {type(key).__name__})"
            )

    # ── Load .env file if declared ─────────────────────────────────
    env_values:    dict[str, str] = {}
    env_file_path: Path | None    = None

    raw_env = db_cfg.get("env_file")
    if raw_env is not None:
        env_file_path = Path(raw_env)
        if not env_file_path.is_absolute():
            env_file_path = (base_dir / env_file_path).resolve()

        if not env_file_path.exists():
            raise FileNotFoundError(
                f"env_file declared in {config_name!r} was not found: "
                f"{env_file_path}\n"
                "Either create the file, fix the path, or remove the "
                "'env_file' key and supply credentials directly in [database]."
            )

        env_values = dotenv_values(env_file_path)   # does NOT mutate os.environ

        if not env_values:
            warnings.warn(
                f"The env_file at {env_file_path} was found but appears to be "
                "empty. Falling back to inline TOML credentials.",
                stacklevel=3,
            )
    else:
        warnings.warn(
            f"No 'env_file' declared in [database] of {config_name!r}. "
            "Reading credentials from the TOML file directly. "
            "Avoid committing plaintext passwords to version control.",
            stacklevel=3,
        )

    # ── Inner resolver — priority chain per key ────────────────────
    def _resolve(key: str, *, required: bool = True, default=None):
        # 1. Live process environment (uppercase by convention)
        value = os.environ.get(key.upper()) or os.environ.get(key)
        if value is not None:
            return value
        # 2. .env file
        value = env_values.get(key.upper()) or env_values.get(key)
        if value is not None:
            return value
        # 3. Inline TOML [database] section
        value = db_cfg.get(key)
        if value is not None:
            return value
        # 4. Fallback or error
        if required:
            sources = []
            if env_file_path:
                sources.append(f"env_file ({env_file_path.name})")
            sources.append(f"[database] in {config_name}")
            sources.append("os.environ")
            raise KeyError(
                f"Required credential {key!r} was not found in any of: "
                + ", ".join(sources)
            )
        return default

    # ── Resolve individual credentials ─────────────────────────────
    host     = _resolve(credentials_keys[0], required=True)
    user     = _resolve(credentials_keys[1], required=True)
    password = _resolve(credentials_keys[2], required=True)
    database = _resolve(credentials_keys[3], required=True)
    port_raw = _resolve(credentials_keys[4], required=False, default=3306)

    # Port coercion — env/os.environ always deliver strings, TOML delivers int
    try:
        port = int(port_raw)
    except (TypeError, ValueError):
        raise TypeError(
            f"'port' must be an integer, got {port_raw!r} "
            f"(type: {type(port_raw).__name__}). "
            "Check the value in your env_file or [database] section."
        )
    if not (1 <= port <= 65535):
        raise ValueError(
            f"'port' value {port} is outside the valid range 1–65535."
        )

    return {
        "host":     host,
        "port":     port,
        "user":     user,
        "password": password,
        "database": database,
    }


# ──────────────────────────────────────────────────────────────────
# HELPER 2 — SQL query
# ──────────────────────────────────────────────────────────────────

def _load_sql(
    query_cfg:   dict,
    base_dir:    Path,
    config_name: str,
) -> str:
    """
    Locate, read, and lightly format the SQL file declared in the
    TOML [query] section.

    Parameters
    ----------
    query_cfg : dict
        Parsed contents of the TOML [query] section.
    base_dir : Path
        Directory of the TOML file, used to resolve relative sql_file paths.
    config_name : str
        TOML filename, used only for error messages.

    Returns
    -------
    str
        Formatted SQL query string, comment-stripped via sqlparse.

    Raises
    ------
    ValueError        – 'sql_file' key is absent from [query].
    FileNotFoundError – declared sql_file path does not exist.
    """

    sql_file_key = query_cfg.get("sql_file")
    if not sql_file_key:
        raise ValueError(
            f"Missing 'sql_file' key in [query] section of {config_name!r}. "
            "Declare the path to the .sql file to use."
        )

    sql_path = Path(sql_file_key)
    if not sql_path.is_absolute():
        sql_path = (base_dir / sql_path).resolve()

    if not sql_path.exists():
        raise FileNotFoundError(
            f"SQL file declared in {config_name!r} was not found: {sql_path}\n"
            "Check the 'sql_file' path under [query]."
        )

    if sql_path.suffix.lower() != ".sql":
        warnings.warn(
            f"Expected a .sql file but got {sql_path.suffix!r}. "
            "Attempting to read anyway.",
            stacklevel=3,
        )

    sql_text = sqlparse.format(
        sql_path.read_text(encoding="utf-8"),
        strip_comments=True,
    ).strip()

    if not sql_text:
        warnings.warn(
            f"The SQL file at {sql_path} is empty. "
            "Queries will fail at execution time.",
            stacklevel=3,
        )

    return sql_text


# ──────────────────────────────────────────────────────────────────
# HELPER 3 — Polygons
# ──────────────────────────────────────────────────────────────────

def _load_polygons(
    polygon_entries: list[dict],
    base_dir:        Path,
) -> dict[str, shapely.geometry.Polygon]:
    """
    Load, validate, and spatially prepare every polygon declared in the
    TOML [[polygons]] array.

    Parameters
    ----------
    polygon_entries : list[dict]
        Parsed contents of raw["polygons"] from the TOML file.
    base_dir : Path
        Directory of the TOML file, used to resolve relative polygon paths.

    Returns
    -------
    dict[str, shapely.geometry.Polygon]
        Mapping of polygon name → prepared Shapely Polygon.
        Polygons marked skip=true are excluded silently (with a warning).

    Raises
    ------
    ValueError        – duplicate polygon name detected.
    FileNotFoundError – a declared polygon path does not exist.
    """

    if not polygon_entries:
        warnings.warn(
            "No [[polygons]] entries found in the configuration. "
            "Spatial checks will not be available.",
            stacklevel=3,
        )
        return {}

    polygon_cache: dict[str, shapely.geometry.Polygon] = {}
    seen_names:    set[str] = set()

    for entry in polygon_entries:
        p_name = entry.get("name")
        p_path = entry.get("path")
        skip   = entry.get("skip", False)

        # ── Required key guards ────────────────────────────────────
        if not p_name:
            warnings.warn(
                "A [[polygons]] entry is missing the 'name' key and will be skipped.",
                stacklevel=3,
            )
            continue
        if not p_path:
            warnings.warn(
                f"Polygon {p_name!r} is missing the 'path' key and will be skipped.",
                stacklevel=3,
            )
            continue

        # ── Duplicate name guard ───────────────────────────────────
        if p_name in seen_names:
            raise ValueError(
                f"Duplicate polygon name {p_name!r} found in [[polygons]]. "
                "Each entry must have a unique 'name'."
            )
        seen_names.add(p_name)

        # ── Skip flag ──────────────────────────────────────────────
        if skip:
            warnings.warn(
                f"Polygon {p_name!r} is marked skip=true and will not be loaded.",
                stacklevel=3,
            )
            continue

        # ── Resolve path ───────────────────────────────────────────
        p_path = Path(p_path)
        if not p_path.is_absolute():
            p_path = (base_dir / p_path).resolve()

        if not p_path.exists():
            raise FileNotFoundError(
                f"Polygon file for {p_name!r} was not found: {p_path}\n"
                "Check the 'path' value in the [[polygons]] entry."
            )

        # ── Load + prepare (spatial index built once here) ─────────
        coords = np.loadtxt(
            str(p_path),
            delimiter = ",",
            skiprows = 1,
            comments  = "#",     # skip any line starting with # anywhere in the file
            )
        polygon = Polygon(coords)
        shapely.prepare(polygon)
        polygon_cache[p_name] = polygon

    return polygon_cache


# ──────────────────────────────────────────────────────────────────
# ORCHESTRATOR — load_config
# ──────────────────────────────────────────────────────────────────

def load_config(
    config_path:      str,
    credentials_keys: tuple[str, str, str, str, str] = (
        "host", "user", "password", "database", "port"
    ),
) -> dict:
    """
    Read and validate a TOML configuration file, resolving all external
    references (env file, SQL file, polygon files) into a single
    self-contained dict ready for use by fetch_seismic_data and
    run_revision_routine.

    Parameters
    ----------
    config_path : str
        Path to the .toml configuration file.
    credentials_keys : tuple[str, str, str, str, str]
        Ordered key names for (host, user, password, database, port).
        Override only when the env file or TOML uses non-standard key names.

    Returns
    -------
    dict with keys:
        "credentials" : dict                          – resolved DB parameters
        "sql"         : str                           – formatted SQL query
        "polygons"    : dict[str, Shapely Polygon]    – prepared polygon cache

    Raises
    ------
    FileNotFoundError – TOML, env_file, sql_file, or a polygon path not found.
    KeyError          – required credential missing from all sources.
    ValueError        – missing TOML sections, duplicate polygon names,
                        or port out of range.
    TypeError         – credentials_key not a string, or port not an integer.
    """

    # ── 1. Load and parse the TOML file ───────────────────────────
    config_path = Path(config_path).resolve()
    if not config_path.exists():
        raise FileNotFoundError(
            f"Configuration file not found: {config_path}"
        )
    if config_path.suffix.lower() != ".toml":
        warnings.warn(
            f"Expected a .toml file but got {config_path.suffix!r}. "
            "Attempting to parse anyway.",
            stacklevel=2,
        )

    with open(config_path, "rb") as fh:
        raw = tomllib.load(fh)

    base_dir    = config_path.parent
    config_name = config_path.name

    # ── 2. Validate required top-level sections ────────────────────
    if "database" not in raw:
        raise ValueError(
            f"Missing [database] section in {config_name!r}. "
            "Please declare connection parameters or an env_file path."
        )
    if "query" not in raw:
        raise ValueError(
            f"Missing [query] section in {config_name!r}. "
            "Please declare 'sql_file'."
        )

    # ── 3. Delegate to the three focused helpers ───────────────────
    credentials = _load_credentials(
        db_cfg           = raw["database"],
        base_dir         = base_dir,
        config_name      = config_name,
        credentials_keys = credentials_keys,
    )

    sql = _load_sql(
        query_cfg   = raw["query"],
        base_dir    = base_dir,
        config_name = config_name,
    )

    polygons = _load_polygons(
        polygon_entries = raw.get("polygons", []),
        base_dir        = base_dir,
    )

    return {
        "credentials": credentials,
        "sql":         sql,
        "polygons":    polygons,
        "checks":      raw.get("checks", [])
    }

In [8]:
# Test the config loader with a sample TOML file
import itertools
if __name__ == "__main__":
    try:
        config = load_config("config.toml", credentials_keys=('SERVER_SC6_HOST', 'SERVER_SC6_USERNAME', 'SERVER_SC6_PASSWORD', 'SERVER_SC6_DATABASE', 'SERVER_SC6_PORT'))
        print("Configuration loaded successfully")
    except Exception as e:
        print(f"Error loading configuration: {e}")

Configuration loaded successfully


## Connecting to the database and executing the query

The next step is to define a function that connects to the database using the credentials from the configuration and executes the SQL query to retrieve the seismic data. This function will be called `fetch_seismic_data` and it will take the configuration dictionary as an argument. It will return a pandas DataFrame with the seismic data retrieved from the database. The function will use the `pymysql` library to establish the connection and execute the query. It will also handle any potential errors that might occur during the connection or query execution, such as authentication errors, connection timeouts, or SQL syntax errors.

In addition, we will consider the case where the user provides more than 1 credential set in the config file (for example, if they want to connect to multiple databases or use different credentials for different queries). In this case, we will allow the user to specify which credential set to use for the connection by passing an additional argument to the function. Also, for incomplete queries that does not have the time range, we will add an optional argument to the function that allows the user to specify the time range for the query (and modify the last part of the query accordingly).

In [9]:
def fetch_seismic_data(
        credentials: dict,
        credentials_keys: tuple[str, str, str, str, str] = ("host", "user", "password", "database", "port"),
        start_time: str | None = None,
        end_time: str | None = None,
        strict_query: str | None = None,
        strict_credentials: dict | None = None,
        **kwargs):
    """
    Connect to the seismic database using the provided credentials and execute the SQL query to fetch seismic data. Optionally, apply a time range filter to the query.

    Parameters
    ----------
    credentials : dict
        Dictionary containing the database connection parameters. Returned by `load_config`.
    credentials_keys: tuple[str]
        List of credentials keys for special cases (HOST, USER, PWD, DB, PORT). Defaults to ("host", "user", "password", "database", "port")
    start_time : str, optional
        Start time for filtering the query results. Any string accepted by :class:`pandas.Timestamp` can be supplied, for example '2024-01-01T00:00:00Z' or '2024-01-01 00:00:00'.

        By setting ``start_time`` and ``end_time``, the function will modify the SQL query to include a WHERE clause that filters the results based on the specified time range. Therefore, the SQL query must not have a time range filter already, otherwise the function will raise an error.

    end_time : str, optional
        End time for filtering the query results. Any string accepted by :class:`pandas.Timestamp` can be supplied, for example '2024-01-01T00:00:00Z' or '2024-01-01 00:00:00'.

    strict_query : str, optional
        If specified, this SQL query will be used instead of the one provided in the configuration. This allows for executing a different query without modifying the config file.

    strict_credentials: dict, optional
        If specified, this dictionary will be used for the database connection instead of the one provided in the configuration. This allows for using different credentials without modifying the config file. It must have the same structure as the `credentials` dictionary returned by `load_config` ('host', 'port', 'user', 'password', 'database' keys).

    **kwargs : dict
        Additional keyword arguments to pass to the :class:`pandas.read_sql` function.

    Returns
    -------
    pd.DataFrame
        Table containing the seismic data retrieved from the database.

    Raises
    -------
    KeyError
        If a required credential is missing from the provided credentials' dictionary.

    TypeError
        If ``start_time`` or ``end_time`` values not follow a valid format, or if ``strict_credentials`` is specified and is not a string, or if credentials is declared, and it is not a dictionary.

    ValueError
        If the SQL query already contains a time range filter and the user tries to apply a new time range filter using ``start_time`` and ``end_time``, or if ``start_time`` is not earlier than ``end_time``, or if the time zones of both values are different.
    """
    # Start by checking start and endtime values
    if (start_time is None and end_time is not None) or (start_time is not None and end_time is None):
        raise TypeError(
            f"start and end time must be provided together. Got start_time={start_time} and end_time={end_time}."
        )
    if start_time and end_time:
        if not isinstance(start_time, str) or not isinstance(end_time, str):
            raise TypeError(
                f"start_time and end_time must be strings, got {type(start_time).__name__} and {type(end_time).__name__} respectively."
            )
        try:
            start_time = pd.to_datetime(start_time)
            end_time = pd.to_datetime(end_time)
        except ValueError:
            raise TypeError(
                f"Invalid start_time or end_time format. "
                "Please provide a valid timestamp string."
            )
        # Check if both values have the same Time Zone
        if start_time.tz != end_time.tz:
            raise ValueError(
                f"start_time ({start_time}) must have the same Time Zone than end_time ({end_time}). Obtained {start_time.tzinfo} and {end_time.tzinfo} respectively."
            )
        if start_time >= end_time:
            raise ValueError(
                f"start_time ({start_time}) must be earlier than end_time ({end_time})."
            )

    if strict_query:
        if not isinstance(strict_query, str):
            raise TypeError(
                f"strict_query must be a string, got {type(strict_query).__name__}."
            )
    if strict_credentials is not None:
        if not isinstance(strict_credentials, dict):
            raise TypeError(
                f"strict_credentials must be a dict, got {type(strict_query).__name__}."
            )
        if not strict_credentials:
            raise ValueError(
                f"strict_credentials must not be empty. Please provide a valid dictionary with the required keys."
            )
        if not {"host", "port", "user", "password", "database"}.issubset(strict_credentials.keys()):
            raise KeyError(
                f"strict_credentials must have the following keys: 'host', 'port', 'user', 'password', 'database'. Got {strict_credentials.keys()}."
            )

    if strict_query is None:
        strict_query = credentials["sql"]

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        strict_query = f"{strict_query} WHERE Origin.time_value BETWEEN '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"

    # Evaluate strict credentials if not None, else let it be
    credentials = strict_credentials if strict_credentials else credentials['credentials']

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        try:
            db_connection = pymysql.connect(
                host = credentials[credentials_keys[0]],
                user = credentials[credentials_keys[1]],
                password = credentials[credentials_keys[2]],
                database = credentials[credentials_keys[3]],
                port = int(credentials.get(credentials_keys[4], 3306)),
            )
        except pymysql.MySQLError as ce:
            raise ConnectionError(
                f"Failed to connect to the database with the provided credentials: {ce}"
            )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                event_table = pd.read_sql(
                    strict_query,
                    db_connection,
                    **kwargs
                )
                pbar.update(1)
        finally:
            db_connection.close()

    return event_table

In [10]:
# Test the fetch_seismic_data function with the loaded configuration
if __name__ == "__main__":
    config = load_config("config.toml", credentials_keys=('SERVER_SC6_HOST', 'SERVER_SC6_USERNAME', 'SERVER_SC6_PASSWORD', 'SERVER_SC6_DATABASE', 'SERVER_SC6_PORT'))
    seismic_data = fetch_seismic_data(config, start_time='2026-03-17', end_time='2026-07-10')
    print("Seismic data fetched successfully. Total events obtained:", len(seismic_data))

Seismic data fetched successfully. Total events obtained: 26571


In [11]:
# Test the fetch_seismic_data function with strict_credentials (It is OK if we get an error, we will use fake credentials)
if __name__ == "__main__":
    try:
        config = load_config("config.toml", credentials_keys=('SERVER_SC3_HOST', 'SERVER_SC3_USERNAME', 'SERVER_SC3_PASSWORD', 'SERVER_SC3_DATABASE', 'SERVER_SC3_PORT'))
        query_example = "SELECT * FROM Origin WHERE Origin.time_value BETWEEN '2026-03-17 00:00:00' AND '2026-07-10 00:00:00' ORDER BY Origin.time_value ASC;"
        other_credentials = {
            "host": "localhost",
            "port": 3306,
            "user": "root",
            "password": "",
            "database": "",
        }
        seismic_data_strict = fetch_seismic_data(config, strict_query=query_example, strict_credentials=other_credentials)
        print("Seismic data fetched successfully with strict credentials. Total events obtained:", len(seismic_data_strict))
    except Exception as e:
        print(f"Error fetching seismic data with strict credentials: {e}")

Error fetching seismic data with strict credentials: Failed to connect to the database with the provided credentials: (2003, "Can't connect to MySQL server on 'localhost' ([Errno 111] Connection refused)")


## Vectorization functions

As we stated in the Vectorization notebook, we can define different vectorization functions that will be used to perform the checks on the seismic data. In general, we have:

In [12]:
# Numeric comparisons
def numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold : float | None= None,
    lower: float | None = None,
    upper: float | None = None,
    dtype = np.float64
) -> np.ndarray:
    """
    Vectorized numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided and equality comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'eq':
        return values == threshold
    elif mode == 'ne':
        return values != threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode!r}")

In [13]:
# Column vs column comparisons
def column_column_mask(
    events: pd.DataFrame,
    left_col: str,
    mode: str,
    right_col: str,
    offset: float = 0.0,
    factor: float = 1.0,
    dtype=np.float64,
) -> np.ndarray:
    """
    Vectorized column to column comparator for seismic quality checks.
    It follows: events[left_col] <<mode>> factor * events[right_col] + offset

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    left_col : str
        Left column to compare.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
    right_col : str
        Right column to compare.
    offset : float, optional
        Offset of the equation, if required. Defaults to zero.
    factor : float, optional
        Multiplier for the right column, if required. Defaults to zero.
    dtype : numpy dtype
        Target dtype for NumPy conversion.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    left = events[left_col].to_numpy(dtype=dtype, copy=False)
    right = events[right_col].to_numpy(dtype=dtype, copy=False) * factor + offset
    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left, right)

In [28]:
# Non-numeric comparisons
def _normalize_text_series(s: pd.Series) -> pd.Series:
    """
    Coerce bytes/bytearray entries to str (UTF-8 decoded) so that a TOML
    string literal like 'DESTACADO' matches regardless of whether the
    source driver returned str or bytes for that row.
    """
    def _to_str(v):
        if isinstance(v, (bytes, bytearray)):
            return v.decode("utf-8", errors="replace")
        return v
    return s.map(_to_str)

def non_numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    values: list[str] | None = None,
) -> np.ndarray:
    """
    Vectorized non-numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'is_null'       -> values that are null/NaN
            'not_null'      -> values that are not null/NaN
            'in'            -> values that are in the provided list
            'not_in'        -> values that are not in the provided list
    values : list[str], optional
        List of values for 'in' or 'not_in' modes.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    s = _normalize_text_series(events[column])  # Normalized text series to handle byte-strings
    if mode == "is_null":
        return s.isna().to_numpy()
    elif mode == "not_null":
        return s.notna().to_numpy()
    elif mode == "in":
        return s.isin(values).to_numpy()
    elif mode == "not_in":
        return (~s.isin(values)).to_numpy()
    else:
        raise ValueError(f"Unsupported category mode: {mode}")

In [15]:
# Polygonal comparison
def build_polygon_mask(
    events: pd.DataFrame,
    lon_col: str,
    lat_col: str,
    polygon: Union[Polygon, shapely.geometry.base.BaseGeometry],
    mode : str = "inside"
) -> np.ndarray:
    """
    Vectorized polygon comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    lon_col : str
        Longitude column to evaluate.
    lat_col : str
        Latitude column to evaluate.
    polygon : shapely.geometry.Polygon or shapely.geometry.base.BaseGeometry
        Shapely polygon to compare.
    mode : str
        Comparison mode:
            'inside'       -> values inside polygon
            'outside'      -> values outside polygon
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    if mode not in ['inside', 'outside']:
        raise ValueError(f"Unsupported polygon mode: {mode}. Accepted modes: 'inside' and 'outside'")
    if not shapely.is_prepared(polygon):
        shapely.prepare(polygon)
    inside = shapely.contains_xy(
        polygon,
        events[lon_col].to_numpy(),
        events[lat_col].to_numpy()
    )
    return inside if mode == "inside" else ~inside

In [16]:
# Temporal comparison
def temporal_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    value: str,
) -> np.ndarray:
    """
    Vectorized temporal comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > Timestamp
            'ge'       -> values >= Timestamp
            'lt'       -> values < Timestamp
            'le'       -> values <= Timestamp
            'eq'       -> values == Timestamp
            'ne'       -> values != Timestamp
    value : str
        Reference timestamp used for the comparison. Any string accepted by
        :class:`pandas.Timestamp` can be supplied, for example
        '2024-01-01T00:00:00Z' or '2024-01-01 00:00:00'.

    Returns
    -------
    numpy.ndarray
        Boolean mask with one entry per row in events. True indicates
        that the row satisfies the requested temporal condition.
    """
    left = pd.to_datetime(events[column], utc=False)
    right = pd.Timestamp(value)

    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left.to_numpy(), right.to_datetime64())

In [17]:
# Composed rules
def combine_masks(
        masks: list[np.ndarray],
        logic: str = "and"
) -> np.ndarray:
    """
    Combine multiple boolean masks using a logical operator.

    Parameters
    ----------
    masks : list[np.ndarray]
        List of boolean masks to combine. All masks must have the same shape.
    logic : str
        Combination logic to apply:
            'and' -> logical AND across all masks
            'or'  -> logical OR across all masks
            'xor' -> logical XOR between exactly 2 masks

    Returns
    -------
    np.ndarray
        Boolean mask with one entry per element in the input masks.

    Raises
    ------
    ValueError
        If no masks are provided, if 'xor' is used with anything other than
        exactly 2 masks, or if an unsupported logic value is supplied.
    """
    if not masks:
        raise ValueError("No masks provided")
    if logic == "and":
        return np.logical_and.reduce(masks)
    elif logic == "or":
        return np.logical_or.reduce(masks)
    elif logic == "xor":
        if len(masks) != 2:
            raise ValueError("XOR logic requires exactly 2 masks")
        return np.logical_xor(masks[0], masks[1])
    else:
        raise ValueError(f"Unsupported logic: {logic!r}")

Then we need to create a function to apply the checks to the seismic data. This function will take the seismic data DataFrame, the list of checks from the configuration, and the polygon cache as inputs. It will iterate over each check, apply the corresponding vectorization function, and combine the results into a final mask that indicates which events are flagged. The function will return a DataFrame with the flagged events with one additional column indicating the reason for the flag. The function will also handle any errors that might occur during the checks, such as missing columns or invalid parameters.

1. Function `run_single_check`: This function will take a single check and apply it to the seismic data. It will return a boolean mask indicating which events are flagged by that check.

In [18]:
def run_single_check(
        events: pd.DataFrame,
        check: dict,
        polygon_cache: dict[str, shapely.geometry.Polygon],
        event_type_col: str = "event_type"
) -> pd.Index:
    """
    Run a single check on the seismic events DataFrame

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic events DataFrame.
    check : dict
        Dictionary containing the check configuration. Usually returned by `load_config` under the 'checks' key.
    polygon_cache : dict[str, shapely.geometry.Polygon]
        Dictionary mapping polygon names to Shapely Polygon objects. Usually returned by `load_config` under the 'polygons' key.
    event_type_col : str
        Name of the column in `events` that contains the event type. Defaults to "event_type".

    Returns
    -----------
    pd.Index
        Index of the flagged events in the original DataFrame.
    """
    # Filter events by event_type at root node
    allowed_types = check.get("event_type")
    if allowed_types:
        if isinstance(allowed_types, str):
            allowed_types = [allowed_types]  # Handle cases where event_type is str
        events = events[events[event_type_col].isin(allowed_types)]

    # If there are no events in events after filtering, return an empty Index Dataframe
    if events.empty:
        return pd.Index([])

    mask = evaluate_node(events, check, polygon_cache)
    flagged = np.where(mask)[0]

    return events.index[flagged]

2. Function `evaluate_node`: This function will recursively evaluate a checks/groups node and return a boolean mask aligned to the subset's row order. It will handle both leaf conditions and nested groups, applying the appropriate vectorization functions and combining the results based on the specified logic. The function will also handle negation of the results if specified in the node.

In [19]:
def evaluate_node(
        subset: pd.DataFrame,
        node: dict,
        polygon_cache: dict[str, shapely.geometry.Polygon]
) -> np.ndarray:
    """
    Recursively evaluate a checks/groups node and return a boolean mask
    aligned to `subset`'s row order.

    A node is either:
      - A pure leaf list holder: has "conditions" (list of condition dicts)
        and/or "groups" (list of nested nodes).
      - If it has exactly ONE child total, "logic" is optional and ignored
        (there is nothing to combine).
      - If it has TWO OR MORE children total, "logic" is required.
      - Optionally negated via node.get("negate", False), regardless of
        child count.

    Parameters
    ----------
    subset : pd.DataFrame
        The subset of events to evaluate against the current node.
    node : dict
        The current checks/groups node to evaluate.
    polygon_cache : dict[str, shapely.geometry.Polygon]
        Dictionary mapping polygon names to Shapely Polygon objects. Usually returned by `load_config` under the 'polygons' key.

    Returns
    -----------
    pd.Index of the flagged events in the original DataFrame.
    """
    child_masks = []

    # Direct leaf conditions attached to this node
    for cond in node.get("conditions", []):
        rule_type = cond.get("rule_type")
        if rule_type not in _CONDITION_DISPATCHERS:
            raise ValueError(
                f"Unsupported rule_type {rule_type!r} in condition {cond}. "
                f"Valid types: {list(_CONDITION_DISPATCHERS.keys())}"
            )
        child_masks.append(_CONDITION_DISPATCHERS[rule_type](subset, cond, polygon_cache))

    # Nested subgroups — recurse
    for group in node.get("groups", []):
        child_masks.append(evaluate_node(subset, group, polygon_cache))

    if not child_masks:
        raise ValueError(
            f"Node {node.get('name', node.get('description', '<unnamed>'))!r} "
            "has no 'conditions' or 'groups' to evaluate."
        )

    # ── Single-child shortcut: logic is irrelevant, skip combine_masks ──
    if len(child_masks) == 1:
        combined = child_masks[0]

        # Warn only if the user redundantly declared a logic key anyway
        if "logic" in node:
            warnings.warn(
                f"Node {node.get('name', node.get('description', '<unnamed>'))!r} "
                f"has a single condition but declares logic={node['logic']!r}. "
                "The 'logic' key is ignored when there is only one child.",
                stacklevel=2,
            )
    else:
        if "logic" not in node:
            raise ValueError(
                f"Node {node.get('name', node.get('description', '<unnamed>'))!r} "
                f"has {len(child_masks)} children (conditions/groups) but no "
                "'logic' key. 'logic' is required when combining 2 or more children."
            )
        combined = combine_masks(child_masks, logic=node["logic"])

    if node.get("negate", False):
        combined = ~combined

    return combined

3. Dispatcher functions: These functions will serve as intermediaries between the check evaluation logic and the vectorization functions. Each dispatcher will handle a specific rule type (numeric, category, temporal, polygon, column_column) and call the corresponding vectorization function with the appropriate parameters extracted from the check configuration. The dispatchers will also handle any necessary preprocessing of the input data before passing it to the vectorization functions.

In [20]:
def _dispatch_numeric(subset: pd.DataFrame, cond: dict, _polygons) -> np.ndarray:
    return numeric_mask(
        events    = subset,
        column    = cond["column"],
        mode      = cond["mode"],
        threshold = cond.get("threshold"),
        lower     = cond.get("lower"),
        upper     = cond.get("upper"),
    )

def _dispatch_category(subset: pd.DataFrame, cond: dict, _polygons) -> np.ndarray:
    values = cond.get("values")
    if values is None and "value" in cond:
        raw_value = cond["value"]
        values = raw_value if isinstance(raw_value, list) else [raw_value]

    return non_numeric_mask(
        events = subset,
        column = cond["column"],
        mode   = cond["mode"],
        values = values,
    )

def _dispatch_temporal(subset: pd.DataFrame, cond: dict, _polygons) -> np.ndarray:
    return temporal_mask(
        events = subset,
        column = cond["column"],
        mode   = cond["mode"],
        value  = cond["value"],
    )

def _dispatch_polygon(subset: pd.DataFrame, cond: dict, polygons: dict) -> np.ndarray:
    polygon_names = cond["polygon"]

    # Normalize: allow both a bare string (legacy) and a list
    if isinstance(polygon_names, str):
        polygon_names = [polygon_names]
    elif not isinstance(polygon_names, list):
        raise TypeError(
            f"'polygon' must be a string or a list of strings, "
            f"got {type(polygon_names).__name__}"
        )

    if not polygon_names:
        raise ValueError("'polygon' list must contain at least one polygon name")

    missing = [p for p in polygon_names if p not in polygons]
    if missing:
        raise KeyError(
            f"Polygon(s) {missing} referenced in a condition were not found "
            f"in the loaded polygon cache. Available: {list(polygons.keys())}"
        )

    mode = cond["mode"]

    per_polygon_masks = [
        build_polygon_mask(
            events  = subset,
            lon_col = cond["lon_col"],
            lat_col = cond["lat_col"],
            polygon = polygons[name],
            mode    = mode,
        )
        for name in polygon_names
    ]

    # Single-polygon shortcut — skip combine_masks entirely
    if len(per_polygon_masks) == 1:
        return per_polygon_masks[0]

    # mode == "inside"  -> event flagged if inside ANY of the listed polygons  (OR)
    # mode == "outside" -> event flagged if outside ALL the listed polygons (AND)
    union_logic = "or" if mode == "inside" else "and"
    return combine_masks(per_polygon_masks, logic=union_logic)

def _dispatch_column_column(subset: pd.DataFrame, cond: dict, _polygons) -> np.ndarray:
    return column_column_mask(
        events    = subset,
        left_col  = cond["left_col"],
        mode      = cond["mode"],
        right_col = cond["right_col"],
        factor    = cond.get("factor", 1.0),
        offset    = cond.get("offset", 0.0),
    )

_CONDITION_DISPATCHERS = {
    "numeric":       _dispatch_numeric,
    "category":      _dispatch_category,
    "temporal":      _dispatch_temporal,
    "polygon":       _dispatch_polygon,
    "column_column": _dispatch_column_column,
}

4. Function `run_checks`: This function will orchestrate the execution of all checks defined in the configuration. It will iterate over each check, call `run_single_check` to evaluate it, and collect the results. The function will then compile a DataFrame of flagged events, including an 'Observations' column that lists which checks were triggered for each event. It will also handle any errors that occur during the execution of individual checks, logging warnings as necessary.

In [21]:
def run_checks(
    events:         pd.DataFrame,
    config_info:         dict,
    event_type_col: str = "event_type",
) -> pd.DataFrame:
    """
    Execute every [[checks]] entry declared in config["checks"] and return
    the flagged rows with an 'Observations' column listing which checks fired.

    Parameters
    ----------
    events : pd.DataFrame
        Full seismic event dataframe.
    config_info : dict
        Output of load_config, extended to include config["checks"]
        (the parsed raw["checks"] list from the TOML).
    event_type_col : str
        Column holding the event type string, used for root-level filtering.

    Returns
    -------
    pd.DataFrame
        Flagged rows with an added 'Observations' column.
    """
    checks   = config_info.get("checks", [])
    polygons = config_info.get("polygons", {})

    if not checks:
        warnings.warn("No [[checks]] entries found in configuration.", stacklevel=2)
        return events.iloc[0:0].copy()

    observations: dict[int, list[str]] = {}

    for check_cfg in checks:
        name = check_cfg.get("name", "<unnamed>")
        try:
            flagged_idx = run_single_check(events, check_cfg, polygons, event_type_col)
        except Exception as exc:
            warnings.warn(f"Check {name!r} raised an error and was skipped: {exc}", stacklevel=2)
            continue

        for idx in flagged_idx:
            observations.setdefault(idx, []).append(name)

    if not observations:
        empty = events.iloc[0:0].copy()
        empty["Observations"] = pd.Series(dtype="object")
        return empty

    flagged = events.loc[sorted(observations.keys())].copy()
    flagged["Observations"] = [", ".join(observations[i]) for i in flagged.index]

    return flagged.reset_index(drop=True)

In [57]:
# Test the run_checks function with the loaded configuration and fetched seismic data
config = load_config("config.toml", credentials_keys=('SERVER_SC6_HOST', 'SERVER_SC6_USERNAME', 'SERVER_SC6_PASSWORD', 'SERVER_SC6_DATABASE', 'SERVER_SC6_PORT'))
starttime_quering = time.time()
seismic_data = fetch_seismic_data(config, start_time='2026-03-17', end_time='2026-07-10')
endtime_quering = time.time()
print(f"Time taken to query seismic data: {endtime_quering - starttime_quering:.2f} seconds")
print(f"Total events obtained: {len(seismic_data)}")

Time taken to query seismic data: 13.54 seconds
Total events obtained: 26635


In [58]:
starttime_checking = time.time()
flagged_events = run_checks(seismic_data, config)
endtime_checking = time.time()
print("Checks executed successfully. Total flagged events:", len(flagged_events))
print(f"Time taken to run checks: {endtime_checking - starttime_checking:.3f} seconds")

Checks executed successfully. Total flagged events: 2075
Time taken to run checks: 0.415 seconds


In [59]:
flagged_events

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment,Observations
0,2026-03-17 00:51:44,SGC2026fhtepm,155.89,1.606607,0.19,3.6,3.818377,3.818377,6.0,6.0,...,not locatable,SGC,"Los Santos - Santander, Colombia",6.826167,-73.084500,MLr_3,Hypo71,RSNC,None,Potentially locatable event
1,2026-03-17 01:10:14,SGC2026fhtunu,146.00,1.791695,0.45,9.1,9.050967,9.050967,6.0,6.0,...,not locatable,SGC,"Los Santos - Santander, Colombia",6.828667,-73.079333,MLr_3,Hypo71,RSNC,None,Potentially locatable event
2,2026-03-17 02:49:01,SGC2026fhxbrc,4.14,0.972878,0.42,722.2,4.454773,4.454773,6.0,6.0,...,not locatable,SGC,"Colombia - Huila, Colombia",3.261833,-74.783500,MLr_2,Hypo71,RSNC,None,Potentially locatable event
3,2026-03-17 02:53:29,SGC2026fhxfnf,158.38,1.562200,0.23,4.3,4.666905,4.666905,6.0,6.0,...,not locatable,SGC,"Los Santos - Santander, Colombia",6.811667,-73.130000,MLr_3,Hypo71,RSNC,None,Potentially locatable event
4,2026-03-17 02:55:12,SGC2026fhxgzw,152.45,1.571655,0.07,1.4,1.414214,1.414214,6.0,6.0,...,not locatable,SGC,"Los Santos - Santander, Colombia",6.819167,-73.065500,MLr_3,Hypo71,RSNC,None,Potentially locatable event
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2070,2026-07-08 17:32:31,SGC2026nilklq,0.00,0.897018,0.59,3.2,4.384062,4.384062,6.0,6.0,...,not locatable,SGC,Northern Colombia,9.559667,-73.547000,MLr_4,Hypo71,RSNC,None,"Locatable event, Potentially locatable event"
2071,2026-07-08 17:36:53,SGC2026nilofo,150.00,1.466561,0.40,9.5,11.667262,11.667262,6.0,6.0,...,not locatable,SGC,"Betulia - Santander, Colombia",6.920500,-73.329667,MLr_vmm,Hypo71,RSNC,None,Potentially locatable event
2072,2026-07-08 17:36:53,SGC2026nilofo,150.00,1.466561,0.40,9.5,11.667262,11.667262,6.0,6.0,...,not locatable,SGC,Northern Colombia,6.920500,-73.329667,MLr_vmm,Hypo71,RSNC,None,Potentially locatable event
2073,2026-07-08 18:30:42,SGC2026ninioz,149.21,1.331011,0.14,2.5,3.959798,3.959798,6.0,6.0,...,not locatable,SGC,"Zapatoca - Santander, Colombia",6.961333,-73.401833,MLr_vmm,Hypo71,RSNC,None,Potentially locatable event


In [26]:
# MAKE A UPDATE FUNCTION THAT RE-READS THE CONFIG FILE AND RE-RUNS THE CHECKS ON THE SAME DATAFRAME, RETURNING THE NEW FLAGGED EVENTS, WITHOUT RE-QUERYING THE DATABASE. IT SHOULD ALSO RETURN THE NEW CONFIGURATION DICTIONARY.